In [1]:
import numpy as np
import scipy.stats as st
import pandas as pd

# --- System Capacity Verification ---
# 1. Load the generated hourly monitoring data
df_hourly = pd.read_csv('../outputs/tables/hourly_monitoring_table_baseline.csv')

print("--- System Capacity Verification ---")

# 2. Prove mathematically that beds_used never exceeded exactly 10
max_beds = df_hourly['beds_used'].max()
print(f"Maximum recorded beds used: {max_beds}")
assert max_beds <= 10, "FAIL: Bed capacity was breached!"
print("PASS: Bed capacity (10) was strictly maintained across all 30 replications.")

# 3. Prove mathematically that queue_length never exceeded exactly 5
max_queue = df_hourly['queue_length'].max()
print(f"Maximum recorded queue length: {max_queue}")
assert max_queue <= 5, "FAIL: Queue capacity was breached!"
print("PASS: Queue capacity (5) was strictly maintained across all 30 replications.")

--- System Capacity Verification ---
Maximum recorded beds used: 10
PASS: Bed capacity (10) was strictly maintained across all 30 replications.
Maximum recorded queue length: 3
PASS: Queue capacity (5) was strictly maintained across all 30 replications.


In [2]:
# --- Patient Routing Verification ---

# 1. Load the generated patient data
df_patients = pd.read_csv('../outputs/tables/patient_level_table_baseline.csv')
print("--- Patient Routing Verification ---")

# 2. Verify Admitted/Discharged Patients
# We filter for patients who successfully completed treatment
df_discharged = df_patients[df_patients['status'] == 'Discharged']

assert df_discharged['bed_number'].notna().all(), "FAIL: A patient is missing a bed number!"
assert df_discharged['admission_time'].notna().all(), "FAIL: A patient is missing an admission time!"
assert df_discharged['discharge_time'].notna().all(), "FAIL: A patient is missing a discharge time!"
print("PASS: All discharged patients have complete bed and timestamp records.")

# 3. Verify Rejected Patients
df_rejected = df_patients[df_patients['status'] == 'Rejected']

assert df_rejected['rejection_reason'].notna().all(), "FAIL: A patient was rejected without a reason!"
assert df_rejected['total_time_in_system'].notna().all(), "FAIL: A rejected patient is missing their time-in-system!"
print("PASS: All rejected patients have properly documented reasons and times.")

--- Patient Routing Verification ---
PASS: All discharged patients have complete bed and timestamp records.
PASS: All rejected patients have properly documented reasons and times.


In [3]:
# --- Bed Overlap Verification ---
print("--- Bed Overlap Verification ---")

# 1. Filter for patients who actually occupied a bed
df_admitted = df_patients[df_patients['status'].isin(['Discharged', 'Censored'])].copy()

# 2. Sort the data by Replication, then Bed Number, then Admission Time
df_sorted = df_admitted.sort_values(by=['replication', 'bed_number', 'admission_time'])

# 3. Shift the discharge times down by 1 row to compare with the next patient's admission
df_sorted['prev_discharge_time'] = df_sorted.groupby(['replication', 'bed_number'])['discharge_time'].shift(1)

# 4. Check for overlaps (rounding to 4 decimal places to prevent floating-point math errors)
# An overlap occurs if a patient is admitted BEFORE the previous patient left
overlaps = df_sorted[df_sorted['admission_time'].round(4) < df_sorted['prev_discharge_time'].round(4)]

assert len(overlaps) == 0, f"FAIL: Found {len(overlaps)} instances of bed double-booking!"
print("PASS: No beds were ever occupied by more than one patient at the same time.")

--- Bed Overlap Verification ---
PASS: No beds were ever occupied by more than one patient at the same time.


In [4]:
import pandas as pd

print("--- Calculating Replication-Level Metrics ---")

# --- Setup Priority Groups ---
def get_priority_group(priority):
    if priority <= 3: return 'Low'
    elif priority <= 7: return 'Medium'
    else: return 'High'

df_patients['priority_group'] = df_patients['priority_level'].apply(get_priority_group)

# 1. Group Patient Data by Replication
rep_patient_stats = []
for rep in range(1, 31):
    df_rep = df_patients[df_patients['replication'] == rep]
    
    total_patients = len(df_rep)
    admitted = df_rep[df_rep['status'].isin(['Admitted', 'Discharged', 'Censored'])]
    rejected = df_rep[df_rep['status'] == 'Rejected']
    discharged = df_rep[df_rep['status'] == 'Discharged']
    
    # Helper to calculate priority rejection rates per replication
    def calc_rej_rate(group_name):
        group_patients = df_rep[df_rep['priority_group'] == group_name]
        if len(group_patients) == 0: return 0.0
        rejections = len(group_patients[group_patients['status'] == 'Rejected'])
        return rejections / len(group_patients)
    
    rep_patient_stats.append({
        'replication': rep,
        'admission_rate': len(admitted) / total_patients,
        'rejection_rate': len(rejected) / total_patients,
        'avg_wait_time': admitted['wait_to_bed'].mean(),
        'avg_length_of_stay': discharged['total_time_in_system'].mean(),
        'low_priority_rejection': calc_rej_rate('Low'),
        'medium_priority_rejection': calc_rej_rate('Medium'),
        'high_priority_rejection': calc_rej_rate('High')
    })

df_rep_patients = pd.DataFrame(rep_patient_stats)

# 2. Group Hourly Data by Replication
rep_hourly_stats = []
for rep in range(1, 31):
    df_rep_h = df_hourly[df_hourly['replication'] == rep]
    
    rep_hourly_stats.append({
        'replication': rep,
        'avg_bed_utilization': df_rep_h['bed_utilization'].mean(),
        'peak_queue_length': df_rep_h['queue_length'].max(),
        'pct_time_queue_full': df_rep_h['queue_full'].mean(), 
        # Updated to perfectly match the rubric definition!
        'pct_time_under_pressure': ((df_rep_h['beds_used'] + df_rep_h['queue_length']) > 10).mean()    
    })

df_rep_hourly = pd.DataFrame(rep_hourly_stats)

# 3. Merge them together into one beautiful master summary table
df_master_summary = pd.merge(df_rep_patients, df_rep_hourly, on='replication')
print("Successfully aggregated all 30 replications including Priority Metrics!")
display(df_master_summary.head())

--- Calculating Replication-Level Metrics ---
Successfully aggregated all 30 replications including Priority Metrics!


,replication,admission_rate,rejection_rate,avg_wait_time,avg_length_of_stay,low_priority_rejection,medium_priority_rejection,high_priority_rejection,avg_bed_utilization,peak_queue_length,pct_time_queue_full,pct_time_under_pressure
0,1,0.979827,0.020173,0.222254,18.753953,0.0,0.030172,0.00,0.846944,2,0.0,0.084722
1,2,0.969613,0.030387,0.327075,19.092915,0.0,0.045833,0.00,0.887222,2,0.0,0.127778
2,3,0.978022,0.021978,0.348867,18.857471,0.0,0.029046,0.25,0.884306,2,0.0,0.118056
3,4,0.967123,0.032877,0.356337,19.110851,0.0,0.047244,0.00,0.889306,3,0.0,0.119444
4,5,0.980447,0.019553,0.249010,18.687136,0.0,0.031250,0.00,0.869444,2,0.0,0.079167


In [5]:
print("--- Final Baseline Confidence Intervals (95%) ---")

def calc_ci(data):
    """Calculates the mean and 95% CI for an array of 30 replication means."""
    mean = np.mean(data)
    sem = st.sem(data) # Standard Error of the Mean
    margin = sem * st.t.ppf((1 + 0.95) / 2., len(data)-1) # t-distribution multiplier
    return mean, mean - margin, mean + margin

# The list of metrics to pull from df_master_summary
metrics_to_report = [
    ('Admission Rate', 'admission_rate', '%'),
    ('Rejection Rate', 'rejection_rate', '%'),
    ('Avg Wait Time (Hours)', 'avg_wait_time', 'hrs'),
    ('Avg Length of Stay (Hours)', 'avg_length_of_stay', 'hrs'),
    ('Avg Bed Utilization', 'avg_bed_utilization', '%'),
    ('Percent Time Queue Full', 'pct_time_queue_full', '%'),
    ('Percent Time Under Pressure', 'pct_time_under_pressure', '%'),
    ('Rejection Rate (Low Priority)', 'low_priority_rejection', '%'),
    ('Rejection Rate (Med Priority)', 'medium_priority_rejection', '%'),
    ('Rejection Rate (High Priority)', 'high_priority_rejection', '%')
]

for title, col, unit in metrics_to_report:
    mean, lower, upper = calc_ci(df_master_summary[col])
    
    if unit == '%':
        print(f"{title}: {mean*100:.2f}%  [95% CI: {lower*100:.2f}% - {upper*100:.2f}%]")
    else:
        print(f"{title}: {mean:.2f} {unit}  [95% CI: {lower:.2f} - {upper:.2f}]")

--- Final Baseline Confidence Intervals (95%) ---
Admission Rate: 97.60%  [95% CI: 97.31% - 97.89%]
Rejection Rate: 2.40%  [95% CI: 2.11% - 2.69%]
Avg Wait Time (Hours): 0.30 hrs  [95% CI: 0.27 - 0.32]
Avg Length of Stay (Hours): 18.85 hrs  [95% CI: 18.77 - 18.93]
Avg Bed Utilization: 87.34%  [95% CI: 86.82% - 87.86%]
Percent Time Queue Full: 0.00%  [95% CI: 0.00% - 0.00%]
Percent Time Under Pressure: 10.39%  [95% CI: 9.42% - 11.35%]
Rejection Rate (Low Priority): 0.22%  [95% CI: 0.06% - 0.38%]
Rejection Rate (Med Priority): 3.47%  [95% CI: 3.05% - 3.89%]
Rejection Rate (High Priority): 1.94%  [95% CI: -0.85% - 4.74%]
